# jevmark: frozen-base baseline B0 (task 1.6)

Thin wrapper: clone the private repo at one commit, install pinned dependencies, build the data, and run `scripts/evaluate.py --ckpt base` for Qwen3-0.6B-Base and Qwen3-1.7B-Base. All logic lives in the repo. Setup and download steps are in `docs/KAGGLE.md`.

Settings: accelerator GPU T4 x2, internet on, secret `GITHUB_TOKEN` attached (a fine-grained, read-only token for this one repository).

In [ ]:
# Parameters: set both before running. COMMIT must be a full 40-character sha.
REPO = "OWNER/jevmark"
COMMIT = "0000000000000000000000000000000000000000"
LIMIT = 30  # records per split in the smoke run

In [ ]:
# Clone REPO at COMMIT. The token reaches git only through environment variables,
# is never put on a command line or in .git/config, and is redacted from any output.
import base64
import os
import re
import subprocess
from pathlib import Path

from kaggle_secrets import UserSecretsClient

assert re.fullmatch(r"[\w.-]+/[\w.-]+", REPO), "REPO must be owner/name"
assert re.fullmatch(r"[0-9a-f]{40}", COMMIT), "COMMIT must be a full 40-character sha"
WORK = Path("/tmp/jevmark")  # outside /kaggle/working, so the notebook output holds only runs/


def run(cmd, env=None, secrets=()):
    result = subprocess.run(cmd, cwd=WORK, env=env, capture_output=True, text=True)
    output = result.stdout + result.stderr
    for secret in secrets:
        output = output.replace(secret, "***")
    if output.strip():
        print(output[-4000:])
    if result.returncode != 0:
        raise RuntimeError(f"{cmd[0]} {cmd[1]} failed with exit code {result.returncode}")


token = UserSecretsClient().get_secret("GITHUB_TOKEN")
header = "AUTHORIZATION: basic " + base64.b64encode(f"x-access-token:{token}".encode()).decode()
git_env = {
    **os.environ,
    "GIT_TERMINAL_PROMPT": "0",
    "GIT_CONFIG_COUNT": "1",
    "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
    "GIT_CONFIG_VALUE_0": header,
}
WORK.mkdir(parents=True, exist_ok=True)
if not (WORK / ".git").exists():
    run(["git", "init", "-q"])
    run(["git", "remote", "add", "origin", f"https://github.com/{REPO}.git"])
run(["git", "fetch", "-q", "--depth", "1", "origin", COMMIT], env=git_env, secrets=(token, header))
run(["git", "checkout", "-q", "--force", "FETCH_HEAD"])
del token, header, git_env

head = subprocess.run(["git", "rev-parse", "HEAD"], cwd=WORK, capture_output=True, text=True, check=True).stdout.strip()
assert head == COMMIT, f"checked out {head}, expected {COMMIT}"
os.chdir(WORK)
print("checked out", head)

In [ ]:
# The Hugging Face packages pinned to uv.lock; Kaggle keeps its own torch and numpy (decision 23). jevmark itself without deps.
!pip install -q -r requirements-kaggle.txt
!pip install -q -e . --no-deps
!python -c "import sys, torch, transformers, peft, datasets; print(sys.version.split()[0], 'torch', torch.__version__, 'cuda', torch.cuda.is_available(), torch.cuda.device_count(), 'transformers', transformers.__version__, 'peft', peft.__version__, 'datasets', datasets.__version__)"

In [ ]:
# Build the eight JSONL splits (about half a minute); the build fails loudly if any check fails.
!make data PY=python

In [ ]:
# Smoke run first: LIMIT records per split on the 0.6B backbone. Writes runs/base_06b_limit{LIMIT}/.
!python scripts/evaluate.py --ckpt base --config configs/base_06b.yaml --limit {LIMIT} --device cuda

In [ ]:
# B0 on Qwen3-0.6B-Base, all eight splits. Writes runs/base_06b/.
!python scripts/evaluate.py --ckpt base --config configs/base_06b.yaml --device cuda

In [ ]:
# B0 on Qwen3-1.7B-Base, all eight splits. Writes runs/base_17b/.
!python scripts/evaluate.py --ckpt base --config configs/base_17b.yaml --device cuda

In [ ]:
# Copy runs/ to /kaggle/working/runs for download, and show what was produced.
import json
import shutil

shutil.copytree(WORK / "runs", "/kaggle/working/runs", dirs_exist_ok=True)
for metrics_path in sorted(Path("/kaggle/working/runs").glob("*/metrics.json")):
    metrics = json.loads(metrics_path.read_text())
    print(metrics_path.parent.name, "commit", metrics["git"]["commit"], "dirty", metrics["git"]["dirty"],
          "fp32 fallback", metrics["precision"]["fp32_fallback_used"], f"{metrics['wall_clock_seconds'] / 60:.1f} min")
    assert metrics["git"]["commit"] == COMMIT